# vLLM Colab Runner & Ngrok Tunnel

This notebook sets up an external vLLM backend server on a Google Colab GPU instance and securely exposes it to your local environment using an `ngrok` tunnel.

**Steps:**
1. Install dependencies (`vllm` and `pyngrok`).
2. Add your ngrok Auth Token.
3. Start the vLLM server with your LoRA weights in the background while opening the ngrok tunnel.

In [ ]:
!pip install -U vllm pyngrok

## 2. Ngrok Authentication
Replace `YOUR_TOKEN_HERE` with your actual ngrok token from your ngrok dashboard.

In [ ]:
!ngrok config add-authtoken YOUR_TOKEN_HERE

## 3. Background vLLM Execution & Ngrok Tunnel
This cell starts the `vllm` server in the background and sets up the ngrok tunnel. Copy the outputted public URL and place it in your local backend's `.env` file as `VLLM_NGROK_URL`.

In [ ]:
import subprocess
import time
import os
import glob
from pyngrok import ngrok

# ── Step 1: Find and prioritize the real libcudart.so.13 ─────────────────────
cu13_lib_dir = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib"

priority_paths = [
    cu13_lib_dir,                                                              # CUDA 13 runtime (real .so.13)
    "/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib",
    "/usr/local/lib/python3.12/dist-packages/nvidia/nvtx/lib",
    "/usr/local/cuda-12.8/targets/x86_64-linux/lib",
    "/usr/local/cuda/lib64",
    "/usr/lib/x86_64-linux-gnu",
]
priority_paths = [p for p in priority_paths if os.path.isdir(p)]

existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(priority_paths) + (":" + existing_ld if existing_ld else "")

# Quick sanity check
test = subprocess.run(
    ["python3", "-c", "import ctypes; ctypes.CDLL('libcudart.so.13'); print('libcudart.so.13 loaded OK')"],
    capture_output=True, text=True, env=os.environ.copy()
)
print(test.stdout.strip() or test.stderr.strip())

# ── Step 2: Remove wrong symlink if it still exists ───────────────────────────
wrong_symlink = "/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.13"
if os.path.islink(wrong_symlink):
    os.remove(wrong_symlink)
    print(f"Removed incorrect symlink: {wrong_symlink}")

# ── Step 3: vLLM command ──────────────────────────────────────────────────────
vllm_command = [
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--enable-lora",
    "--max-lora-rank", "32",
    "--max-model-len", "2048",
    "--trust-remote-code",
    "--gpu-memory-utilization", "0.85",
    "--lora-modules", "jobs_lora=abdoghazala7/Jobs",
    "--enforce-eager",
    "--attention-backend", "TRITON_ATTN",
]

# ── Step 4: Launch vLLM ───────────────────────────────────────────────────────
print("\nStarting vLLM server...")
process = subprocess.Popen(
    vllm_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

# ── Step 5: Wait for server — T4 needs up to 10 min for CUDA graph capture ───
print("Waiting for model to load (up to 10 min on T4 due to CUDA graph compilation)...\n")

# These strings appear in vLLM logs when the server is fully ready
READY_SIGNALS = [
    "Uvicorn running on",
    "Application startup complete",
    "vllm server started",
]

# These strings indicate a fatal crash — no point waiting further
FATAL_SIGNALS = [
    "ImportError",
    "RuntimeError",
    "CUDA error",
    "out of memory",
    "Traceback (most recent call last)",
]

server_ready = False
fatal_error  = False
timeout      = 600          # 10 minutes — enough for T4 CUDA graph capture
start_time   = time.time()

while time.time() - start_time < timeout:
    line = process.stdout.readline()
    if not line:
        if process.poll() is not None:
            print("\nProcess exited unexpectedly.")
            break
        time.sleep(0.1)
        continue

    decoded = line.decode("utf-8").strip()
    print(decoded)

    if any(sig in decoded for sig in READY_SIGNALS):
        server_ready = True
        print("\nServer is ready!")
        break

    if any(sig in decoded for sig in FATAL_SIGNALS):
        print(f"\nFatal error detected — stopping early.")
        fatal_error = True
        break

# ── Step 6: Open ngrok tunnel ─────────────────────────────────────────────────
if server_ready:
    public_url = ngrok.connect(8000).public_url
    print("\n" + "=" * 50)
    print("SUCCESS! vLLM is running and publicly accessible.")
    print(f"Ngrok URL : {public_url}")
    print(f"\nAdd this to your .env file:")
    print(f"VLLM_NGROK_URL={public_url}")
    print("=" * 50 + "\n")

    try:
        for line in iter(process.stdout.readline, b""):
            print(line.decode("utf-8").strip())
    except KeyboardInterrupt:
        print("Shutting down...")
        process.terminate()
        ngrok.kill()
else:
    elapsed = int(time.time() - start_time)
    print(f"\nServer did not become ready after {elapsed}s.")
    remaining = process.stdout.read(4096)
    if remaining:
        print(remaining.decode("utf-8"))
    process.terminate()